In [6]:
import os, sys, time, json, shutil, subprocess, warnings
warnings.filterwarnings("ignore")

try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
if shutil.which("java") is None:
    subprocess.run("apt-get -qq install -y openjdk-17-jdk-headless > /dev/null", shell=True, check=True)

from google.colab import drive
drive.mount("/content/drive")
PERSIST_DIR = "/content/drive/MyDrive/7006SCN_GSOD"
SCRATCH_DIR = "/content/gsod_scratch"
LOCAL_DIR   = "/content/gsod_local"

def sp(path):
    """Path passed to Spark (identical to the local path on Colab)."""
    return path

RAW_DIR     = SCRATCH_DIR + "/raw_csv"
ARCHIVE_DIR = PERSIST_DIR + "/archives"
PARQUET_RAW = LOCAL_DIR + "/parquet/gsod_raw"
FEATURES    = LOCAL_DIR + "/parquet/gsod_features"
PRED_DIR    = LOCAL_DIR + "/parquet/predictions"
MODELS_DIR  = LOCAL_DIR + "/models"
OUT_DIR     = PERSIST_DIR + "/outputs"
for d in [PERSIST_DIR, SCRATCH_DIR, LOCAL_DIR, ARCHIVE_DIR, MODELS_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

def backup_to_drive(local_path):
    dest = local_path.replace(LOCAL_DIR, PERSIST_DIR)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(local_path, dest)
    print("backed up ->", dest)

def restore_from_drive(local_path):
    if not os.path.exists(local_path):
        shutil.copytree(local_path.replace(LOCAL_DIR, PERSIST_DIR), local_path)
        print("restored <-", local_path)

YEARS       = [2020, 2021, 2022, 2023]
TRAIN_YEARS = [2020, 2021, 2022]
TEST_YEAR   = 2023
SEED        = 42

from pyspark.sql import SparkSession, functions as F, Window
from pyspark import StorageLevel
TOTAL_RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
DRIVER_MEM = f"{max(2, int(TOTAL_RAM_GB * 0.7))}g"   # leave ~30% for Python/OS

spark = (SparkSession.builder
         .master("local[*]")                                    # use all Colab CPU cores
         .appName("7006SCN_GSOD_Task2")
         .config("spark.driver.memory", DRIVER_MEM)
         .config("spark.sql.shuffle.partitions", "64")          # default 200 is too many for a few cores
         .config("spark.sql.adaptive.enabled", "true")          # AQE: coalesces small shuffle partitions at runtime
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
         .config("spark.ui.showConsoleProgress", "false")       # keep notebook output clean
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def n_partitions(df):
    return df.rdd.getNumPartitions()

def safe_persist(df, name, level=StorageLevel.MEMORY_AND_DISK):
    """Persist and materialise immediately (count) so later timings exclude the caching cost."""
    out = df.persist(level)
    out.count()
    return out

def save_csv(pdf, name):
    path = f"{OUT_DIR}/{name}"
    pdf.to_csv(path, index=False)
    print("saved ->", path)

print(f"Spark {spark.version} | master={spark.sparkContext.master} | cores={spark.sparkContext.defaultParallelism} "
      f"| driver memory={DRIVER_MEM} | RAM={TOTAL_RAM_GB:.1f} GB")
print("PERSIST_DIR =", PERSIST_DIR)

restore_from_drive(PARQUET_RAW)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.4 | master=local[*] | cores=2 | driver memory=8g | RAM=12.7 GB
PERSIST_DIR = /content/drive/MyDrive/7006SCN_GSOD


In [7]:
raw = spark.read.parquet(sp(PARQUET_RAW))
print(f"Partitions after ingestion: {n_partitions(raw)}")
print(f"Rows: {raw.count():,} | Columns: {len(raw.columns)}")

Partitions after ingestion: 127
Rows: 16,179,201 | Columns: 29


In [8]:
import pandas as pd
SENTINELS = {"TEMP": 9999.9, "DEWP": 9999.9, "SLP": 9999.9, "STP": 9999.9, "MAX": 9999.9, "MIN": 9999.9,
             "VISIB": 999.9, "WDSP": 999.9, "MXSPD": 999.9, "GUST": 999.9, "SNDP": 999.9, "PRCP": 99.99}

def is_missing(c, v):
    return F.col(c).isNull() | (F.abs(F.col(c) - v) < 1e-6)

before = raw.agg(*[F.sum(is_missing(c, v).cast("long")).alias(c) for c, v in SENTINELS.items()]).toPandas().T

df = raw
for c, v in SENTINELS.items():
    df = df.withColumn(c, F.when(is_missing(c, v), None).otherwise(F.col(c)))
funnel = [("1. Raw rows", raw.count())]

In [9]:
df = df.dropna(subset=["STATION", "DATE", "LATITUDE", "LONGITUDE", "TEMP"])
funnel.append(("2. Required fields present", df.count()))

def valid(c, lo, hi):
    return F.when((F.col(c) >= lo) & (F.col(c) <= hi), F.col(c))

df = (df.withColumn("TEMP", valid("TEMP", -100, 135))
        .withColumn("DEWP", valid("DEWP", -120, 100))
        .withColumn("MAX", valid("MAX", -100, 140))
        .withColumn("MIN", valid("MIN", -110, 130))
        .withColumn("SLP", valid("SLP", 850, 1090))
        .withColumn("VISIB", valid("VISIB", 0, 100))
        .withColumn("WDSP", valid("WDSP", 0, 150))
        .withColumn("MXSPD", valid("MXSPD", 0, 200))
        .withColumn("GUST", valid("GUST", 0, 250))
        .withColumn("PRCP", valid("PRCP", 0, 50))
        .withColumn("SNDP", valid("SNDP", 0, 500))
        .withColumn("MAX", F.when(F.col("MIN").isNull() | (F.col("MAX") >= F.col("MIN")), F.col("MAX")))
        .dropna(subset=["TEMP"])
        .drop("STP", "STP_COUNT"))
funnel.append(("3. Physically valid TEMP", df.count()))

df = df.dropDuplicates(["STATION", "DATE"])
funnel.append(("4. De-duplicated station-days", df.count()))

after = df.agg(*[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in SENTINELS if c in df.columns]).toPandas().T
missing = before.join(after, how="left", lsuffix="_b", rsuffix="_a").reset_index()
missing.columns = ["column", "missing_before", "missing_after_cleaning"]
missing["pct_before"] = (100 * missing.missing_before / funnel[0][1]).round(2)
print(missing.to_string(index=False))
save_csv(missing, "task2_missing_values.csv")

column  missing_before  missing_after_cleaning  pct_before
  TEMP               0                     0.0        0.00
  DEWP          764666                762645.0        4.73
   SLP         5595337               5545425.0       34.58
   STP               0                     NaN        0.00
   MAX           10690                 10614.0        0.07
   MIN           12478                 12454.0        0.08
 VISIB         4271082               4263149.0       26.40
  WDSP          611841                609638.0        3.78
 MXSPD          831429                829124.0        5.14
  GUST        11737729              11718995.0       72.55
  SNDP        15063696              15012356.0       93.11
  PRCP         1246835               1236179.0        7.71
saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/task2_missing_values.csv


In [5]:
N_PART = max(64, spark.sparkContext.defaultParallelism * 8)
print(f"Partitions before repartition: {n_partitions(df)}")
df = df.repartition(N_PART, "STATION")
print(f"Partitions after repartition(\"STATION\"): {n_partitions(df)}")

Partitions before repartition: 32
Partitions after repartition("STATION"): 64


In [10]:
for i, name in enumerate(["FOG", "RAIN", "SNOW", "HAIL", "THUNDER", "TORNADO"]):
    df = df.withColumn(name, F.coalesce(F.substring("FRSHTT", i + 1, 1).cast("int"), F.lit(0)))

df = (df.withColumn("EXT_HEAT",  F.coalesce((F.col("MAX") >= 95).cast("int"), F.lit(0)))
        .withColumn("EXT_COLD",  F.coalesce((F.col("MIN") <= -4).cast("int"), F.lit(0)))
        .withColumn("EXT_RAIN",  F.coalesce((F.col("PRCP") >= 1.0).cast("int"), F.lit(0)))
        .withColumn("EXT_WIND",  F.coalesce(((F.col("MXSPD") >= 34) | (F.col("GUST") >= 50)).cast("int"), F.lit(0)))
        .withColumn("EXT_STORM", F.greatest("HAIL", "THUNDER", "TORNADO"))
        .withColumn("IS_EXTREME", F.greatest("EXT_HEAT", "EXT_COLD", "EXT_RAIN", "EXT_WIND", "EXT_STORM")))

df = (df.withColumn("TEMP_RANGE", F.col("MAX") - F.col("MIN"))           # diurnal range
        .withColumn("DEWP_DEPRESSION", F.col("TEMP") - F.col("DEWP"))    # dryness / humidity proxy
        .withColumn("GUST_MISSING", F.col("GUST").isNull().cast("int"))
        .withColumn("SNDP", F.coalesce("SNDP", F.lit(0.0))))

df = (df.withColumn("MONTH", F.month("DATE"))
        .withColumn("DOY", F.dayofyear("DATE"))
        .withColumn("DOY_SIN", F.sin(2 * 3.141592653589793 * F.col("DOY") / 365.25))   # cyclical encoding:
        .withColumn("DOY_COS", F.cos(2 * 3.141592653589793 * F.col("DOY") / 365.25))   # 31 Dec is next to 1 Jan
        .withColumn("HEMISPHERE", F.when(F.col("LATITUDE") >= 0, "North").otherwise("South"))
        .withColumn("SEASON_N", F.when(F.col("MONTH").isin(12, 1, 2), "Winter")
                                 .when(F.col("MONTH").isin(3, 4, 5), "Spring")
                                 .when(F.col("MONTH").isin(6, 7, 8), "Summer").otherwise("Autumn"))
        .withColumn("SEASON", F.when(F.col("HEMISPHERE") == "North", F.col("SEASON_N"))
                               .otherwise(F.expr("CASE SEASON_N WHEN 'Winter' THEN 'Summer' WHEN 'Summer' THEN 'Winter' "
                                                 "WHEN 'Spring' THEN 'Autumn' ELSE 'Spring' END")))
        .drop("SEASON_N")
        .withColumn("CLIMATE_ZONE", F.when(F.abs("LATITUDE") < 23.5, "Tropical")
                                     .when(F.abs("LATITUDE") < 35, "Subtropical")
                                     .when(F.abs("LATITUDE") < 55, "Temperate")
                                     .otherwise("Polar/Subpolar")))

In [11]:
df = df.withColumn("DAY_NUM", F.datediff("DATE", F.lit("1970-01-01")))
w      = Window.partitionBy("STATION").orderBy("DATE")
w_3day = Window.partitionBy("STATION").orderBy("DAY_NUM").rangeBetween(-2, 0)   # true 3 calendar days

prev_day_ok = F.datediff("DATE", F.lag("DATE").over(w)) == 1
df = (df.withColumn("TEMP_CHANGE_1D", F.when(prev_day_ok, F.col("TEMP") - F.lag("TEMP").over(w)))
        .withColumn("SLP_CHANGE_1D",  F.when(prev_day_ok, F.col("SLP") - F.lag("SLP").over(w)))   # pressure tendency
        .withColumn("TEMP_MEAN_3D",   F.avg("TEMP").over(w_3day))
        .withColumn("PRCP_SUM_3D",    F.sum("PRCP").over(w_3day))
        .withColumn("EXTREME_COUNT_3D", F.sum("IS_EXTREME").over(w_3day))
        .withColumn("NEXT_DATE", F.lead("DATE").over(w))
        .withColumn("NEXT_IS_EXTREME", F.lead("IS_EXTREME").over(w)))

df = (df.filter(F.datediff("NEXT_DATE", "DATE") == 1)
        .withColumn("label", F.col("NEXT_IS_EXTREME").cast("double"))
        .drop("NEXT_DATE", "NEXT_IS_EXTREME", "DAY_NUM"))

In [12]:
NUM_FEATURES = ["LATITUDE", "LONGITUDE", "ELEVATION", "TEMP", "DEWP", "SLP", "VISIB", "WDSP", "MXSPD", "GUST",
                "MAX", "MIN", "PRCP", "SNDP", "TEMP_RANGE", "DEWP_DEPRESSION", "TEMP_CHANGE_1D", "SLP_CHANGE_1D",
                "TEMP_MEAN_3D", "PRCP_SUM_3D", "EXTREME_COUNT_3D", "DOY_SIN", "DOY_COS"]
BIN_FEATURES = ["FOG", "RAIN", "SNOW", "HAIL", "THUNDER", "TORNADO", "IS_EXTREME", "GUST_MISSING"]
CAT_FEATURES = ["SEASON", "HEMISPHERE", "CLIMATE_ZONE"]
ID_COLS      = ["STATION", "NAME", "DATE", "YEAR", "MONTH",
                "EXT_HEAT", "EXT_COLD", "EXT_RAIN", "EXT_WIND", "EXT_STORM"]

features = df.select(*ID_COLS, *[F.col(c).cast("double") for c in NUM_FEATURES], *BIN_FEATURES, *CAT_FEATURES, "label")

t0 = time.perf_counter()
features.write.mode("overwrite").partitionBy("YEAR").parquet(sp(FEATURES))
backup_to_drive(FEATURES)
print(f"Feature table written in {time.perf_counter()-t0:,.0f}s")

features = spark.read.parquet(sp(FEATURES))
n_final = features.count()
funnel.append(("5. With next-day label (modelling rows)", n_final))
funnel_df = pd.DataFrame(funnel, columns=["stage", "rows"])
funnel_df["pct_of_raw"] = (100 * funnel_df.rows / funnel_df.rows.iloc[0]).round(2)
print(funnel_df.to_string(index=False))
save_csv(funnel_df, "task2_pipeline_funnel.csv")
print(f"Partitions of feature table on read: {n_partitions(features)} | columns: {len(features.columns)}")

backed up -> /content/drive/MyDrive/7006SCN_GSOD/parquet/gsod_features
Feature table written in 476s
                                  stage     rows  pct_of_raw
                            1. Raw rows 16179201      100.00
             2. Required fields present 16127943       99.68
               3. Physically valid TEMP 16127810       99.68
          4. De-duplicated station-days 16127810       99.68
5. With next-day label (modelling rows) 15899103       98.27
saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/task2_pipeline_funnel.csv
Partitions of feature table on read: 7 | columns: 45


In [13]:
balance = (features.withColumn("split", F.when(F.col("YEAR") == TEST_YEAR, "test").otherwise("train"))
           .groupBy("split", "label").count().orderBy("split", "label").toPandas())
balance["share_pct"] = (100 * balance["count"] / balance.groupby("split")["count"].transform("sum")).round(2)
print(balance.to_string(index=False))
save_csv(balance, "task2_class_balance.csv")

types = features.agg(*[F.sum(c).alias(c) for c in ["EXT_HEAT", "EXT_COLD", "EXT_RAIN", "EXT_WIND", "EXT_STORM", "IS_EXTREME"]]).toPandas().T
types.columns = ["days"]
print(types)

split  label    count  share_pct
 test    0.0  3342044      84.41
 test    1.0   617126      15.59
train    0.0 10154253      85.04
train    1.0  1785680      14.96
saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/task2_class_balance.csv
               days
EXT_HEAT     931081
EXT_COLD     521147
EXT_RAIN     239981
EXT_WIND     137378
EXT_STORM    723136
IS_EXTREME  2401997


In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, StandardScaler

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_IDX", handleInvalid="keep") for c in CAT_FEATURES]
encoder  = OneHotEncoder(inputCols=[f"{c}_IDX" for c in CAT_FEATURES],
                         outputCols=[f"{c}_OHE" for c in CAT_FEATURES], handleInvalid="keep")
imputer  = Imputer(inputCols=NUM_FEATURES, outputCols=[f"{c}_IMP" for c in NUM_FEATURES], strategy="median")
assembler = VectorAssembler(inputCols=[f"{c}_IMP" for c in NUM_FEATURES] + BIN_FEATURES + [f"{c}_OHE" for c in CAT_FEATURES],
                            outputCol="features_raw", handleInvalid="keep")
scaler   = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=[*indexers, encoder, imputer, assembler, scaler])

train = features.filter(F.col("YEAR").isin(TRAIN_YEARS))
t0 = time.perf_counter()
prep_model = prep_pipeline.fit(train)
print(f"Pipeline fitted on training years {TRAIN_YEARS} in {time.perf_counter()-t0:,.0f}s")
prep_model.write().overwrite().save(sp(f"{MODELS_DIR}/prep_pipeline"))
backup_to_drive(f"{MODELS_DIR}/prep_pipeline")
for i, s in enumerate(prep_model.stages, 1):
    print(f"  stage {i}: {type(s).__name__}")

Pipeline fitted on training years [2020, 2021, 2022] in 232s
backed up -> /content/drive/MyDrive/7006SCN_GSOD/models/prep_pipeline
  stage 1: StringIndexerModel
  stage 2: StringIndexerModel
  stage 3: StringIndexerModel
  stage 4: OneHotEncoderModel
  stage 5: ImputerModel
  stage 6: VectorAssembler
  stage 7: StandardScalerModel


In [15]:
sample_out = prep_model.transform(train.limit(1000))
attrs = sample_out.schema["features_raw"].metadata["ml_attr"]["attrs"]
all_attrs = sorted((a for group in attrs.values() for a in group), key=lambda a: a["idx"])
feature_names = [a["name"].replace("_IMP", "") for a in all_attrs]
json.dump(feature_names, open(f"{OUT_DIR}/feature_names.json", "w"))
print(len(feature_names), "features:", feature_names)
sample_out.select("features", "label").show(3, truncate=100)

44 features: ['LATITUDE', 'LONGITUDE', 'ELEVATION', 'TEMP', 'DEWP', 'SLP', 'VISIB', 'WDSP', 'MXSPD', 'GUST', 'MAX', 'MIN', 'PRCP', 'SNDP', 'TEMP_RANGE', 'DEWP_DEPRESSION', 'TEMP_CHANGE_1D', 'SLP_CHANGE_1D', 'TEMP_MEAN_3D', 'PRCP_SUM_3D', 'EXTREME_COUNT_3D', 'DOY_SIN', 'DOY_COS', 'FOG', 'RAIN', 'SNOW', 'HAIL', 'THUNDER', 'TORNADO', 'IS_EXTREME', 'GUST_MISSING', 'SEASON_OHE_Spring', 'SEASON_OHE_Autumn', 'SEASON_OHE_Summer', 'SEASON_OHE_Winter', 'SEASON_OHE___unknown', 'HEMISPHERE_OHE_North', 'HEMISPHERE_OHE_South', 'HEMISPHERE_OHE___unknown', 'CLIMATE_ZONE_OHE_Temperate', 'CLIMATE_ZONE_OHE_Subtropical', 'CLIMATE_ZONE_OHE_Polar/Subpolar', 'CLIMATE_ZONE_OHE_Tropical', 'CLIMATE_ZONE_OHE___unknown']
+----------------------------------------------------------------------------------------------------+-----+
|                                                                                            features|label|
+------------------------------------------------------------------------------

In [16]:
DESC = {
 "STATION": "WMO/DATSAV3 + WBAN station id", "NAME": "Station name", "DATE": "Observation date", "YEAR": "Year (partition key)",
 "MONTH": "Month 1-12", "LATITUDE": "Degrees", "LONGITUDE": "Degrees", "ELEVATION": "Metres",
 "TEMP": "Mean temperature (F)", "DEWP": "Mean dew point (F)", "SLP": "Mean sea-level pressure (mb)",
 "VISIB": "Mean visibility (miles)", "WDSP": "Mean wind speed (kn)", "MXSPD": "Max sustained wind (kn)",
 "GUST": "Max gust (kn)", "MAX": "Max temperature (F)", "MIN": "Min temperature (F)", "PRCP": "Precipitation (in)",
 "SNDP": "Snow depth (in), missing -> 0", "TEMP_RANGE": "MAX - MIN", "DEWP_DEPRESSION": "TEMP - DEWP (dryness)",
 "TEMP_CHANGE_1D": "TEMP today - yesterday", "SLP_CHANGE_1D": "Pressure tendency (mb/day)",
 "TEMP_MEAN_3D": "3-day mean TEMP", "PRCP_SUM_3D": "3-day precipitation total", "EXTREME_COUNT_3D": "Extreme days in last 3",
 "DOY_SIN": "sin(day of year)", "DOY_COS": "cos(day of year)", "FOG": "FRSHTT fog flag", "RAIN": "FRSHTT rain flag",
 "SNOW": "FRSHTT snow flag", "HAIL": "FRSHTT hail flag", "THUNDER": "FRSHTT thunder flag", "TORNADO": "FRSHTT tornado flag",
 "IS_EXTREME": "Today is an extreme day (rules in Task 1)", "GUST_MISSING": "1 if no gust reported",
 "EXT_HEAT": "MAX>=95F", "EXT_COLD": "MIN<=-4F", "EXT_RAIN": "PRCP>=1in", "EXT_WIND": "MXSPD>=34kn or GUST>=50kn",
 "EXT_STORM": "hail/thunder/tornado", "SEASON": "Meteorological season (hemisphere-aware)", "HEMISPHERE": "North/South",
 "CLIMATE_ZONE": "Latitude band", "label": "TARGET: tomorrow is an extreme day (1/0)"}
dd = pd.DataFrame([{"column": f.name, "type": f.dataType.simpleString(), "description": DESC.get(f.name, ""),
                    "role": "target" if f.name == "label" else "feature" if f.name in NUM_FEATURES + BIN_FEATURES + CAT_FEATURES else "id/analysis"}
                   for f in features.schema.fields])
save_csv(dd, "data_dictionary.csv")
dd

saved -> /content/drive/MyDrive/7006SCN_GSOD/outputs/data_dictionary.csv


,column,type,description,role
0,STATION,string,WMO/DATSAV3 + WBAN station id,id/analysis
1,NAME,string,Station name,id/analysis
2,DATE,date,Observation date,id/analysis
3,MONTH,int,Month 1-12,id/analysis
4,EXT_HEAT,int,MAX>=95F,id/analysis
5,EXT_COLD,int,MIN<=-4F,id/analysis
6,EXT_RAIN,int,PRCP>=1in,id/analysis
7,EXT_WIND,int,MXSPD>=34kn or GUST>=50kn,id/analysis
8,EXT_STORM,int,hail/thunder/tornado,id/analysis
9,LATITUDE,double,Degrees,feature
